In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# =========================
# Cell 1: Load dataset
# 作用：
# 1. 读取已经合并好的 4h 新闻 topic 特征 + 油价数据
# 2. 设置核心 topic
# 3. 检查目标变量分布
# =========================

import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("/content/drive/MyDrive/bigquery")
DATA_FILE = DATA_DIR / "direction_dataset_4h_topic_count_only.csv"

df4 = pd.read_csv(DATA_FILE)
df4["timestamp_utc"] = pd.to_datetime(df4["timestamp_utc"], utc=True)
df4 = df4.sort_values("timestamp_utc").reset_index(drop=True)

# 核心 topic：只保留和美伊/油价事件逻辑最相关的 topic
core_topics = [
    "inventory_eia",
    "diplomacy_ceasefire",
    "military_escalation",
    "hormuz_shipping",
    "sanctions_exports"
]

print("Dataset shape:", df4.shape)
print("\nTarget distribution:")
print(df4["target_direction"].value_counts(normalize=True))
display(df4.head())

Dataset shape: (297, 111)

Target distribution:
target_direction
1    0.575758
0    0.424242
Name: proportion, dtype: float64


,timestamp_utc,wti_close,wti_ret_0,wti_ret_lag1,wti_ret_lag2,wti_ret_lag3,wti_ret_lag6,wti_ret_lag12,wti_vol_3,wti_vol_6,...,log1p_sanctions_exports_source_count,log1p_demand_china_growth_relevance_sum,log1p_diplomacy_ceasefire_relevance_sum,log1p_hormuz_shipping_relevance_sum,log1p_inventory_eia_relevance_sum,log1p_macro_dollar_fed_relevance_sum,log1p_military_escalation_relevance_sum,log1p_opec_gulf_supply_relevance_sum,log1p_other_oil_relevant_relevance_sum,log1p_sanctions_exports_relevance_sum
0,2026-02-15 20:00:00+00:00,62.900002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.609438,1.386294,4.770685,2.708050,3.218876,3.713572,3.433987,4.248495,0.000000
1,2026-02-16 00:00:00+00:00,62.910000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.098612,1.386294,2.197225,4.828314,2.197225,3.806662,3.688879,3.871201,4.007333,1.609438
2,2026-02-16 04:00:00+00:00,62.910000,0.000000,0.000159,NaN,NaN,NaN,NaN,NaN,NaN,...,1.791759,1.098612,2.708050,3.583519,4.574711,3.970292,3.218876,4.204693,4.369448,2.639057
3,2026-02-17 04:00:00+00:00,63.200001,0.004599,0.000000,0.000159,NaN,NaN,NaN,0.002611,NaN,...,1.945910,1.945910,2.944439,3.688879,3.091042,4.025352,3.218876,4.007333,4.465908,2.708050
4,2026-02-17 08:00:00+00:00,63.669998,0.007409,0.004599,0.000000,0.000159,NaN,NaN,0.003740,NaN,...,2.197225,1.609438,3.258097,4.927254,2.833213,4.094345,4.499810,4.663439,4.304065,2.890372


In [5]:
# =========================
# Cell 2: Identify core topic features
# 作用：
# 1. 找出所有 core topic 相关的特征
# 2. 不使用 all_news、other_oil_relevant、macro 等杂项 topic
# =========================

exclude_cols = [
    "timestamp_utc",
    "wti_close",
    "future_ret_next",
    "target_direction"
]

numeric_cols = [
    c for c in df4.columns
    if c not in exclude_cols
    and pd.api.types.is_numeric_dtype(df4[c])
]

def is_topic_feature(col, topic):
    return col.startswith(topic + "_") or col.startswith("log1p_" + topic + "_")

def get_topic_current_cols(df, topic):
    """
    提取某个 topic 的当前窗口特征。
    不包括 lag，不包括 shock。
    """
    return [
        c for c in df.columns
        if is_topic_feature(c, topic)
        and not c.endswith("_lag1")
        and not c.endswith("_lag2")
        and not c.endswith("_lag3")
        and not c.endswith("_shock")
        and pd.api.types.is_numeric_dtype(df[c])
    ]

topic_current_cols = {
    topic: get_topic_current_cols(df4, topic)
    for topic in core_topics
}

core_count_cols = []
for topic in core_topics:
    core_count_cols.extend(topic_current_cols[topic])

core_count_cols = list(dict.fromkeys(core_count_cols))

print("Core count features:", len(core_count_cols))
for topic in core_topics:
    print(topic, len(topic_current_cols[topic]))
    print(topic_current_cols[topic])

Core count features: 35
inventory_eia 7
['inventory_eia_count', 'inventory_eia_source_count', 'inventory_eia_relevance_sum', 'inventory_eia_relevance_mean', 'log1p_inventory_eia_count', 'log1p_inventory_eia_source_count', 'log1p_inventory_eia_relevance_sum']
diplomacy_ceasefire 7
['diplomacy_ceasefire_count', 'diplomacy_ceasefire_source_count', 'diplomacy_ceasefire_relevance_sum', 'diplomacy_ceasefire_relevance_mean', 'log1p_diplomacy_ceasefire_count', 'log1p_diplomacy_ceasefire_source_count', 'log1p_diplomacy_ceasefire_relevance_sum']
military_escalation 7
['military_escalation_count', 'military_escalation_source_count', 'military_escalation_relevance_sum', 'military_escalation_relevance_mean', 'log1p_military_escalation_count', 'log1p_military_escalation_source_count', 'log1p_military_escalation_relevance_sum']
hormuz_shipping 7
['hormuz_shipping_count', 'hormuz_shipping_source_count', 'hormuz_shipping_relevance_sum', 'hormuz_shipping_relevance_mean', 'log1p_hormuz_shipping_count', '

In [6]:
# =========================
# Cell 3: Build topic shock features
# 作用：
# 把原始 topic count 转换为“异常新闻强度”
#
# 逻辑：
# shock = 当前值相对过去 rolling window 的 z-score
#
# 避免数据泄漏：
# rolling mean/std 使用 shift(1)，只看过去，不看当前和未来
# =========================

df_feat = df4.copy()

def rolling_zscore_past(series, window=12, min_periods=4):
    """
    用过去窗口计算 z-score。
    window=12 表示过去 12 个 4h 窗口，也就是过去约 48 小时。
    """
    past_mean = series.shift(1).rolling(window=window, min_periods=min_periods).mean()
    past_std = series.shift(1).rolling(window=window, min_periods=min_periods).std()

    z = (series - past_mean) / past_std.replace(0, np.nan)
    z = z.replace([np.inf, -np.inf], np.nan).fillna(0)
    return z

shock_cols = []

for col in core_count_cols:
    # 只对 count / source_count / relevance_sum / log1p 特征做 shock
    if (
        col.endswith("_count")
        or col.endswith("_source_count")
        or col.endswith("_relevance_sum")
        or col.startswith("log1p_")
    ):
        shock_col = f"{col}_shock"
        df_feat[shock_col] = rolling_zscore_past(
            df_feat[col],
            window=12,
            min_periods=4
        )
        shock_cols.append(shock_col)

print("Shock features:", len(shock_cols))
print(shock_cols[:30])

Shock features: 30
['inventory_eia_count_shock', 'inventory_eia_source_count_shock', 'inventory_eia_relevance_sum_shock', 'log1p_inventory_eia_count_shock', 'log1p_inventory_eia_source_count_shock', 'log1p_inventory_eia_relevance_sum_shock', 'diplomacy_ceasefire_count_shock', 'diplomacy_ceasefire_source_count_shock', 'diplomacy_ceasefire_relevance_sum_shock', 'log1p_diplomacy_ceasefire_count_shock', 'log1p_diplomacy_ceasefire_source_count_shock', 'log1p_diplomacy_ceasefire_relevance_sum_shock', 'military_escalation_count_shock', 'military_escalation_source_count_shock', 'military_escalation_relevance_sum_shock', 'log1p_military_escalation_count_shock', 'log1p_military_escalation_source_count_shock', 'log1p_military_escalation_relevance_sum_shock', 'hormuz_shipping_count_shock', 'hormuz_shipping_source_count_shock', 'hormuz_shipping_relevance_sum_shock', 'log1p_hormuz_shipping_count_shock', 'log1p_hormuz_shipping_source_count_shock', 'log1p_hormuz_shipping_relevance_sum_shock', 'sanctio

In [7]:
# =========================
# Cell 4: Build signed pressure features
# 作用：
# 把不同 topic 按经济含义组合成油价压力指数。
#
# 逻辑：
# military_escalation / hormuz_shipping / sanctions_exports -> 油价上行压力
# diplomacy_ceasefire -> 油价下行压力
# inventory_eia -> 单独作为库存冲击
#
# 为什么这样做：
# 地缘政治新闻中的“负面情绪”不一定让油价下跌。
# 战争/制裁/航运风险通常可能推高油价。
# 停火/外交缓和通常可能压低风险溢价。
# =========================

def safe_col(df, col):
    if col in df.columns:
        return df[col]
    return pd.Series(0, index=df.index)

def topic_shock_signal(df, topic):
    """
    对一个 topic，把几个核心 shock 特征平均成一个 topic signal。
    优先使用 log1p 的 shock 特征。
    """
    candidate_cols = [
        f"log1p_{topic}_count_shock",
        f"log1p_{topic}_source_count_shock",
        f"log1p_{topic}_relevance_sum_shock",
        f"{topic}_count_shock",
        f"{topic}_source_count_shock",
        f"{topic}_relevance_sum_shock",
    ]

    existing = [c for c in candidate_cols if c in df.columns]

    if len(existing) == 0:
        return pd.Series(0, index=df.index)

    return df[existing].mean(axis=1)

# 每个 topic 的 shock signal
df_feat["military_signal"] = topic_shock_signal(df_feat, "military_escalation")
df_feat["hormuz_signal"] = topic_shock_signal(df_feat, "hormuz_shipping")
df_feat["sanctions_signal"] = topic_shock_signal(df_feat, "sanctions_exports")
df_feat["diplomacy_signal"] = topic_shock_signal(df_feat, "diplomacy_ceasefire")
df_feat["inventory_signal"] = topic_shock_signal(df_feat, "inventory_eia")

# 上行风险压力
df_feat["geo_risk_pressure"] = (
    df_feat["military_signal"]
    + df_feat["hormuz_signal"]
    + df_feat["sanctions_signal"]
) / 3

# 下行缓和压力
df_feat["diplomacy_relief_pressure"] = df_feat["diplomacy_signal"]

# 库存冲击
df_feat["inventory_shock_pressure"] = df_feat["inventory_signal"]

# 净地缘政治油价压力
df_feat["net_geo_oil_pressure"] = (
    df_feat["geo_risk_pressure"]
    - df_feat["diplomacy_relief_pressure"]
)

# 核心新闻总注意力：使用 log1p_count 类特征
attention_cols = [
    c for c in core_count_cols
    if c.startswith("log1p_") and c.endswith("_count")
]

df_feat["core_news_attention"] = 0
for c in attention_cols:
    df_feat["core_news_attention"] += df_feat[c]

engineered_cols = [
    "military_signal",
    "hormuz_signal",
    "sanctions_signal",
    "diplomacy_signal",
    "inventory_signal",
    "geo_risk_pressure",
    "diplomacy_relief_pressure",
    "inventory_shock_pressure",
    "net_geo_oil_pressure",
    "core_news_attention"
]

print("Engineered pressure features:", engineered_cols)
display(df_feat[engineered_cols].describe())

Engineered pressure features: ['military_signal', 'hormuz_signal', 'sanctions_signal', 'diplomacy_signal', 'inventory_signal', 'geo_risk_pressure', 'diplomacy_relief_pressure', 'inventory_shock_pressure', 'net_geo_oil_pressure', 'core_news_attention']


,military_signal,hormuz_signal,sanctions_signal,diplomacy_signal,inventory_signal,geo_risk_pressure,diplomacy_relief_pressure,inventory_shock_pressure,net_geo_oil_pressure,core_news_attention
count,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000,297.000000
mean,-0.033156,0.097065,0.022726,-0.045233,0.029869,0.028878,-0.045233,0.029869,0.074111,32.763638
std,1.426724,1.306020,1.253644,1.303756,1.339766,1.012501,1.303756,1.339766,1.447421,4.569087
min,-6.011823,-4.050436,-4.820782,-3.757932,-5.152848,-3.879419,-3.757932,-5.152848,-5.605729,16.973133
25%,-0.760268,-0.723302,-0.772155,-0.853210,-0.708972,-0.565281,-0.853210,-0.708972,-0.790082,29.995166
50%,-0.053712,-0.005704,0.000000,-0.017169,0.000000,0.058110,-0.017169,0.000000,-0.000332,33.809861
75%,0.736611,0.822297,0.706129,0.744918,0.701768,0.593549,0.744918,0.701768,0.923450,36.234583
max,11.688100,5.799744,5.292033,6.152233,6.903176,4.677226,6.152233,6.903176,7.942668,43.727364


In [8]:
# =========================
# Cell 5: Build interaction features
# 作用：
# 检查新闻压力是否在市场状态不同的时候作用更强。
#
# 例子：
# net_geo_pressure × WTI volatility
# net_geo_pressure × recent WTI return
# =========================

interaction_cols = []

if "wti_vol_3" in df_feat.columns:
    df_feat["net_geo_pressure_x_wti_vol3"] = (
        df_feat["net_geo_oil_pressure"] * df_feat["wti_vol_3"]
    )
    interaction_cols.append("net_geo_pressure_x_wti_vol3")

if "wti_vol_6" in df_feat.columns:
    df_feat["net_geo_pressure_x_wti_vol6"] = (
        df_feat["net_geo_oil_pressure"] * df_feat["wti_vol_6"]
    )
    interaction_cols.append("net_geo_pressure_x_wti_vol6")

if "wti_ret_lag1" in df_feat.columns:
    df_feat["net_geo_pressure_x_wti_ret_lag1"] = (
        df_feat["net_geo_oil_pressure"] * df_feat["wti_ret_lag1"]
    )
    interaction_cols.append("net_geo_pressure_x_wti_ret_lag1")

if "wti_ret_lag2" in df_feat.columns:
    df_feat["net_geo_pressure_x_wti_ret_lag2"] = (
        df_feat["net_geo_oil_pressure"] * df_feat["wti_ret_lag2"]
    )
    interaction_cols.append("net_geo_pressure_x_wti_ret_lag2")

print("Interaction features:", interaction_cols)

Interaction features: ['net_geo_pressure_x_wti_vol3', 'net_geo_pressure_x_wti_vol6', 'net_geo_pressure_x_wti_ret_lag1', 'net_geo_pressure_x_wti_ret_lag2']


In [9]:
# =========================
# Cell 6: Define meaningful move filtering
# 作用：
# 去掉未来 4h return 绝对值很小的样本。
#
# filter_q = 0.25 表示去掉最小 25% 的价格变化。
# filter_q = 0.40 表示去掉最小 40% 的价格变化。
#
# 注意：
# 这会改变任务定义：
# 从预测所有微小涨跌
# 变成预测较明显的上涨/下跌。
# =========================

def make_meaningful_df(df, filter_q=0.25):
    temp = df.copy()

    if filter_q > 0:
        threshold = temp["future_ret_next"].abs().quantile(filter_q)
        temp = temp[temp["future_ret_next"].abs() >= threshold].copy()
    else:
        threshold = 0.0

    temp["target_direction"] = (temp["future_ret_next"] > 0).astype(int)

    return temp, threshold

filter_q_list = [0.0, 0.20, 0.25, 0.30, 0.40]

for q in filter_q_list:
    temp, th = make_meaningful_df(df_feat, filter_q=q)
    print("=" * 60)
    print("filter_q:", q)
    print("return threshold:", th)
    print("shape:", temp.shape)
    print(temp["target_direction"].value_counts(normalize=True))

filter_q: 0.0
return threshold: 0.0
shape: (297, 155)
target_direction
1    0.575758
0    0.424242
Name: proportion, dtype: float64
filter_q: 0.2
return threshold: 0.0027836518781697
shape: (237, 155)
target_direction
1    0.607595
0    0.392405
Name: proportion, dtype: float64
filter_q: 0.25
return threshold: 0.004142015839454
shape: (223, 155)
target_direction
1    0.600897
0    0.399103
Name: proportion, dtype: float64
filter_q: 0.3
return threshold: 0.00497492811083558
shape: (208, 155)
target_direction
1    0.600962
0    0.399038
Name: proportion, dtype: float64
filter_q: 0.4
return threshold: 0.006786346287320701
shape: (178, 155)
target_direction
1    0.606742
0    0.393258
Name: proportion, dtype: float64


In [10]:
# =========================
# Cell 7: Define models
# 作用：
# 跑简单、可解释的模型：
# 1. Logistic L1 / L2
# 2. LinearSVC
# 3. RidgeClassifier
#
# 这些比 XGBoost 更适合小样本，也更容易解释。
# =========================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC

def make_model(model_name):
    if model_name.startswith("Logit_L1"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="liblinear",
                C=C,
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("Logit_L2"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=C,
                max_iter=5000,
                class_weight=None
            ))
        ])

    if model_name.startswith("LinearSVC"):
        C = float(model_name.split("_C")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                C=C,
                max_iter=10000,
                random_state=42
            ))
        ])

    if model_name.startswith("Ridge"):
        alpha = float(model_name.split("_alpha")[-1])
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", RidgeClassifier(alpha=alpha))
        ])

    raise ValueError(model_name)

In [11]:
# =========================
# Cell 8: Walk-forward evaluation
# 作用：
# 用 TimeSeriesSplit 做时间序列验证。
# 不随机打乱，避免未来信息泄漏。
#
# 输出：
# accuracy
# balanced_accuracy
# AUC
# up_rate_pred
# =========================

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

def get_model_score(model, X):
    """
    不同模型的 score 输出方式不同：
    - Logistic 有 predict_proba
    - LinearSVC / Ridge 有 decision_function
    """
    if hasattr(model, "predict_proba"):
        try:
            return model.predict_proba(X)[:, 1]
        except:
            pass

    if hasattr(model, "decision_function"):
        try:
            return model.decision_function(X)
        except:
            pass

    return model.predict(X)

def evaluate_walkforward(df, feature_cols, model_name, n_splits=5):
    data = df.sort_values("timestamp_utc").reset_index(drop=True).copy()

    X = data[feature_cols].replace([np.inf, -np.inf], np.nan)
    y = data["target_direction"].astype(int).values

    tscv = TimeSeriesSplit(n_splits=n_splits)

    y_true_all = []
    y_pred_all = []
    y_score_all = []

    for train_idx, test_idx in tscv.split(X):
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        if len(np.unique(y_train)) < 2:
            continue

        model = make_model(model_name)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)
        score = get_model_score(model, X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(pred)
        y_score_all.extend(score)

    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_score_all = np.array(y_score_all)

    try:
        auc = roc_auc_score(y_true_all, y_score_all)
    except:
        auc = np.nan

    return {
        "model": model_name,
        "n_features": len(feature_cols),
        "accuracy": accuracy_score(y_true_all, y_pred_all),
        "balanced_accuracy": balanced_accuracy_score(y_true_all, y_pred_all),
        "f1": f1_score(y_true_all, y_pred_all, zero_division=0),
        "auc": auc,
        "up_rate_true": y_true_all.mean(),
        "up_rate_pred": y_pred_all.mean(),
        "n_eval": len(y_true_all),
        "confusion_matrix": confusion_matrix(y_true_all, y_pred_all).tolist()
    }

In [12]:
# =========================
# Cell 9: Feature ablation
# 作用：
# 比较不同文本特征工程是否提升预测效果。
#
# 对比：
# 1. Core count only
# 2. Topic shock only
# 3. Signed pressure only
# 4. Count + shock
# 5. Count + shock + pressure
# 6. Count + shock + pressure + interaction
# =========================

feature_sets_ablation = {
    "Core count only": core_count_cols,
    "Topic shock only": shock_cols,
    "Signed pressure only": engineered_cols,
    "Signed pressure + interaction": engineered_cols + interaction_cols,
    "Core count + shock": core_count_cols + shock_cols,
    "Core count + pressure": core_count_cols + engineered_cols,
    "Core count + shock + pressure": core_count_cols + shock_cols + engineered_cols,
    "Core count + shock + pressure + interaction": (
        core_count_cols + shock_cols + engineered_cols + interaction_cols
    )
}

models_to_run = []

for C in [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]:
    models_to_run.append(f"Logit_L1_C{C}")
    models_to_run.append(f"Logit_L2_C{C}")

for C in [0.01, 0.05, 0.1, 0.5]:
    models_to_run.append(f"LinearSVC_C{C}")

for alpha in [0.5, 1.0, 2.0, 5.0, 10.0]:
    models_to_run.append(f"Ridge_alpha{alpha}")

ablation_rows = []

for filter_q in filter_q_list:
    df_run, return_threshold = make_meaningful_df(df_feat, filter_q=filter_q)

    print("\n" + "=" * 100)
    print("Running feature ablation | filter_q:", filter_q)
    print("Shape:", df_run.shape)
    print("Return threshold:", return_threshold)

    for feature_set_name, cols in feature_sets_ablation.items():
        cols = list(dict.fromkeys(cols))

        for model_name in models_to_run:
            print(f"q={filter_q} | {feature_set_name} | {model_name}")

            try:
                res = evaluate_walkforward(
                    df_run,
                    feature_cols=cols,
                    model_name=model_name,
                    n_splits=5
                )

                res["filter_q"] = filter_q
                res["return_threshold"] = return_threshold
                res["feature_set"] = feature_set_name
                ablation_rows.append(res)

            except Exception as e:
                print("Failed:", filter_q, feature_set_name, model_name, e)

ablation_results = pd.DataFrame(ablation_rows)

ablation_results = ablation_results.sort_values(
    ["accuracy", "balanced_accuracy", "auc"],
    ascending=False
)

display(
    ablation_results[
        [
            "filter_q",
            "feature_set",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ].head(40)
)

ablation_results.to_csv(DATA_DIR / "experiment1_feature_ablation.csv", index=False)
print("Saved:", DATA_DIR / "experiment1_feature_ablation.csv")


Running feature ablation | filter_q: 0.0
Shape: (297, 155)
Return threshold: 0.0
q=0.0 | Core count only | Logit_L1_C0.01
q=0.0 | Core count only | Logit_L2_C0.01
q=0.0 | Core count only | Logit_L1_C0.05
q=0.0 | Core count only | Logit_L2_C0.05
q=0.0 | Core count only | Logit_L1_C0.1
q=0.0 | Core count only | Logit_L2_C0.1
q=0.0 | Core count only | Logit_L1_C0.3
q=0.0 | Core count only | Logit_L2_C0.3
q=0.0 | Core count only | Logit_L1_C0.5
q=0.0 | Core count only | Logit_L2_C0.5
q=0.0 | Core count only | Logit_L1_C1.0
q=0.0 | Core count only | Logit_L2_C1.0
q=0.0 | Core count only | LinearSVC_C0.01
q=0.0 | Core count only | LinearSVC_C0.05
q=0.0 | Core count only | LinearSVC_C0.1
q=0.0 | Core count only | LinearSVC_C0.5
q=0.0 | Core count only | Ridge_alpha0.5
q=0.0 | Core count only | Ridge_alpha1.0
q=0.0 | Core count only | Ridge_alpha2.0
q=0.0 | Core count only | Ridge_alpha5.0
q=0.0 | Core count only | Ridge_alpha10.0
q=0.0 | Topic shock only | Logit_L1_C0.01
q=0.0 | Topic shock 

,filter_q,feature_set,model,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
684,0.40,Core count only,LinearSVC_C0.01,35,0.655172,0.601575,0.609649,0.606897,0.772414,0.006786
810,0.40,Core count + shock + pressure,LinearSVC_C0.01,75,0.648276,0.583533,0.607656,0.606897,0.820690,0.006786
509,0.30,Core count only,Logit_L2_C0.1,35,0.647059,0.583333,0.613033,0.600000,0.835294,0.004975
507,0.30,Core count only,Logit_L2_C0.05,35,0.647059,0.573529,0.616061,0.600000,0.882353,0.004975
176,0.20,Core count only,Logit_L1_C0.5,35,0.646154,0.576766,0.593000,0.605128,0.846154,0.002784
323,0.20,Core count + shock + pressure + interaction,Logit_L1_C0.5,79,0.646154,0.574510,0.601695,0.605128,0.856410,0.002784
178,0.20,Core count only,Logit_L1_C1.0,35,0.646154,0.574510,0.598833,0.605128,0.856410,0.002784
260,0.20,Core count + shock,Logit_L1_C0.5,65,0.646154,0.572254,0.601255,0.605128,0.866667,0.002784
302,0.20,Core count + shock + pressure,Logit_L1_C0.5,75,0.646154,0.572254,0.601145,0.605128,0.866667,0.002784
258,0.20,Core count + shock,Logit_L1_C0.3,65,0.646154,0.569998,0.595972,0.605128,0.876923,0.002784


Saved: /content/drive/MyDrive/bigquery/experiment1_feature_ablation.csv


In [13]:
# =========================
# Cell 10: Meaningful move sensitivity summary
# 作用：
# 看不同 filter_q 下最佳结果是否稳定。
#
# 这个表适合放 PPT：
# filter_q 越高，说明去掉越多小波动。
# 如果 0.20 / 0.25 / 0.30 / 0.40 都提升，说明结论更稳。
# =========================

best_by_filter = (
    ablation_results
    .sort_values(["filter_q", "accuracy", "balanced_accuracy", "auc"], ascending=[True, False, False, False])
    .groupby("filter_q")
    .head(5)
)

display(
    best_by_filter[
        [
            "filter_q",
            "feature_set",
            "model",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred",
            "return_threshold"
        ]
    ]
)

best_by_filter.to_csv(DATA_DIR / "experiment2_best_by_filter_q.csv", index=False)
print("Saved:", DATA_DIR / "experiment2_best_by_filter_q.csv")

,filter_q,feature_set,model,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred,return_threshold
6,0.00,Core count only,Logit_L1_C0.3,0.616327,0.557931,0.550276,0.591837,0.828571,0.000000
111,0.00,Core count + pressure,Logit_L1_C0.3,0.600000,0.544138,0.540345,0.591837,0.812245,0.000000
1,0.00,Core count only,Logit_L2_C0.01,0.600000,0.525517,0.530759,0.591837,0.910204,0.000000
68,0.00,Signed pressure + interaction,Logit_L2_C0.1,0.595918,0.556207,0.549655,0.591837,0.726531,0.000000
5,0.00,Core count only,Logit_L2_C0.1,0.591837,0.527931,0.516621,0.591837,0.853061,0.000000
176,0.20,Core count only,Logit_L1_C0.5,0.646154,0.576766,0.593000,0.605128,0.846154,0.002784
323,0.20,Core count + shock + pressure + interaction,Logit_L1_C0.5,0.646154,0.574510,0.601695,0.605128,0.856410,0.002784
178,0.20,Core count only,Logit_L1_C1.0,0.646154,0.574510,0.598833,0.605128,0.856410,0.002784
260,0.20,Core count + shock,Logit_L1_C0.5,0.646154,0.572254,0.601255,0.605128,0.866667,0.002784
302,0.20,Core count + shock + pressure,Logit_L1_C0.5,0.646154,0.572254,0.601145,0.605128,0.866667,0.002784


Saved: /content/drive/MyDrive/bigquery/experiment2_best_by_filter_q.csv


In [14]:
# =========================
# Cell 11: Single-topic-only sensitivity
# 作用：
# 看每个 topic 单独是否有预测力。
#
# 问题：
# 只用 inventory_eia 能不能预测？
# 只用 diplomacy_ceasefire 能不能预测？
# 只用 military_escalation 能不能预测？
# =========================

def get_topic_shock_cols(topic):
    return [
        c for c in shock_cols
        if is_topic_feature(c, topic)
    ]

topic_feature_versions = {}

for topic in core_topics:
    current = topic_current_cols[topic]
    shock = get_topic_shock_cols(topic)

    topic_feature_versions[(topic, "current")] = current
    topic_feature_versions[(topic, "shock")] = shock
    topic_feature_versions[(topic, "current+shock")] = current + shock

topic_rows = []

topic_models = [
    "Logit_L2_C0.1",
    "Logit_L1_C0.5",
    "LinearSVC_C0.05"
]

topic_filter_q_list = [0.0, 0.25, 0.30, 0.40]

for filter_q in topic_filter_q_list:
    df_run, return_threshold = make_meaningful_df(df_feat, filter_q=filter_q)

    for topic in core_topics:
        for version in ["current", "shock", "current+shock"]:
            cols = topic_feature_versions[(topic, version)]
            cols = list(dict.fromkeys(cols))

            if len(cols) == 0:
                continue

            for model_name in topic_models:
                print(f"single topic | q={filter_q} | {topic} | {version} | {model_name}")

                try:
                    res = evaluate_walkforward(
                        df_run,
                        feature_cols=cols,
                        model_name=model_name,
                        n_splits=5
                    )

                    res["filter_q"] = filter_q
                    res["return_threshold"] = return_threshold
                    res["topic"] = topic
                    res["feature_version"] = version
                    res["analysis"] = "single_topic"
                    topic_rows.append(res)

                except Exception as e:
                    print("Failed:", filter_q, topic, version, model_name, e)

single_topic_results = pd.DataFrame(topic_rows)

single_topic_results = single_topic_results.sort_values(
    ["filter_q", "accuracy", "balanced_accuracy", "auc"],
    ascending=[True, False, False, False]
)

display(
    single_topic_results[
        [
            "filter_q",
            "topic",
            "feature_version",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].head(60)
)

single_topic_results.to_csv(DATA_DIR / "experiment3a_single_topic_sensitivity.csv", index=False)
print("Saved:", DATA_DIR / "experiment3a_single_topic_sensitivity.csv")

single topic | q=0.0 | inventory_eia | current | Logit_L2_C0.1
single topic | q=0.0 | inventory_eia | current | Logit_L1_C0.5
single topic | q=0.0 | inventory_eia | current | LinearSVC_C0.05
single topic | q=0.0 | inventory_eia | shock | Logit_L2_C0.1
single topic | q=0.0 | inventory_eia | shock | Logit_L1_C0.5
single topic | q=0.0 | inventory_eia | shock | LinearSVC_C0.05
single topic | q=0.0 | inventory_eia | current+shock | Logit_L2_C0.1
single topic | q=0.0 | inventory_eia | current+shock | Logit_L1_C0.5
single topic | q=0.0 | inventory_eia | current+shock | LinearSVC_C0.05
single topic | q=0.0 | diplomacy_ceasefire | current | Logit_L2_C0.1
single topic | q=0.0 | diplomacy_ceasefire | current | Logit_L1_C0.5
single topic | q=0.0 | diplomacy_ceasefire | current | LinearSVC_C0.05
single topic | q=0.0 | diplomacy_ceasefire | shock | Logit_L2_C0.1
single topic | q=0.0 | diplomacy_ceasefire | shock | Logit_L1_C0.5
single topic | q=0.0 | diplomacy_ceasefire | shock | LinearSVC_C0.05
sin

,filter_q,topic,feature_version,model,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred
6,0.00,inventory_eia,current+shock,Logit_L2_C0.1,13,0.591837,0.517069,0.489586,0.591837,0.910204
7,0.00,inventory_eia,current+shock,Logit_L1_C0.5,13,0.591837,0.513966,0.505276,0.591837,0.926531
1,0.00,inventory_eia,current,Logit_L1_C0.5,7,0.591837,0.510862,0.515966,0.591837,0.942857
12,0.00,diplomacy_ceasefire,shock,Logit_L2_C0.1,6,0.587755,0.530690,0.495517,0.591837,0.816327
8,0.00,inventory_eia,current+shock,LinearSVC_C0.05,13,0.583673,0.505517,0.472069,0.591837,0.926531
0,0.00,inventory_eia,current,Logit_L2_C0.1,7,0.583673,0.502414,0.506310,0.591837,0.942857
13,0.00,diplomacy_ceasefire,shock,Logit_L1_C0.5,6,0.579592,0.523793,0.496897,0.591837,0.808163
14,0.00,diplomacy_ceasefire,shock,LinearSVC_C0.05,6,0.575510,0.528103,0.508759,0.591837,0.763265
23,0.00,military_escalation,shock,LinearSVC_C0.05,6,0.575510,0.509483,0.503793,0.591837,0.861224
2,0.00,inventory_eia,current,LinearSVC_C0.05,7,0.575510,0.495517,0.469621,0.591837,0.934694


Saved: /content/drive/MyDrive/bigquery/experiment3a_single_topic_sensitivity.csv


In [15]:
# =========================
# Cell 12: Leave-one-topic-out sensitivity
# 作用：
# 用全部 core topics，然后每次删掉一个 topic。
#
# 解释：
# 如果删掉某个 topic 后 AUC / accuracy 明显下降，
# 说明这个 topic 很重要。
# =========================

def combine_topic_cols(topics, use_shock=True):
    cols = []

    for topic in topics:
        cols.extend(topic_current_cols[topic])

        if use_shock:
            cols.extend(get_topic_shock_cols(topic))

    return list(dict.fromkeys(cols))

leave_one_rows = []

loo_filter_q_list = [0.25, 0.30, 0.40]
loo_models = [
    "Logit_L2_C0.1",
    "Logit_L1_C0.5",
    "LinearSVC_C0.05"
]

for filter_q in loo_filter_q_list:
    df_run, return_threshold = make_meaningful_df(df_feat, filter_q=filter_q)

    for model_name in loo_models:

        # baseline: all core topics
        all_cols = combine_topic_cols(core_topics, use_shock=True)

        base_res = evaluate_walkforward(
            df_run,
            feature_cols=all_cols,
            model_name=model_name,
            n_splits=5
        )

        base_res["filter_q"] = filter_q
        base_res["return_threshold"] = return_threshold
        base_res["analysis"] = "all_core"
        base_res["removed_topic"] = "none"
        base_res["remaining_topics"] = ",".join(core_topics)
        leave_one_rows.append(base_res)

        # leave one topic out
        for removed_topic in core_topics:
            remaining_topics = [t for t in core_topics if t != removed_topic]
            cols = combine_topic_cols(remaining_topics, use_shock=True)

            print(f"leave one out | q={filter_q} | remove={removed_topic} | {model_name}")

            try:
                res = evaluate_walkforward(
                    df_run,
                    feature_cols=cols,
                    model_name=model_name,
                    n_splits=5
                )

                res["filter_q"] = filter_q
                res["return_threshold"] = return_threshold
                res["analysis"] = "leave_one_out"
                res["removed_topic"] = removed_topic
                res["remaining_topics"] = ",".join(remaining_topics)
                leave_one_rows.append(res)

            except Exception as e:
                print("Failed:", filter_q, removed_topic, model_name, e)

leave_one_results = pd.DataFrame(leave_one_rows)

display(
    leave_one_results[
        [
            "filter_q",
            "analysis",
            "removed_topic",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].sort_values(["filter_q", "model", "analysis", "accuracy"], ascending=[True, True, True, False])
)

leave_one_results.to_csv(DATA_DIR / "experiment3b_leave_one_topic_out.csv", index=False)
print("Saved:", DATA_DIR / "experiment3b_leave_one_topic_out.csv")

leave one out | q=0.25 | remove=inventory_eia | Logit_L2_C0.1
leave one out | q=0.25 | remove=diplomacy_ceasefire | Logit_L2_C0.1
leave one out | q=0.25 | remove=military_escalation | Logit_L2_C0.1
leave one out | q=0.25 | remove=hormuz_shipping | Logit_L2_C0.1
leave one out | q=0.25 | remove=sanctions_exports | Logit_L2_C0.1
leave one out | q=0.25 | remove=inventory_eia | Logit_L1_C0.5
leave one out | q=0.25 | remove=diplomacy_ceasefire | Logit_L1_C0.5
leave one out | q=0.25 | remove=military_escalation | Logit_L1_C0.5
leave one out | q=0.25 | remove=hormuz_shipping | Logit_L1_C0.5
leave one out | q=0.25 | remove=sanctions_exports | Logit_L1_C0.5
leave one out | q=0.25 | remove=inventory_eia | LinearSVC_C0.05
leave one out | q=0.25 | remove=diplomacy_ceasefire | LinearSVC_C0.05
leave one out | q=0.25 | remove=military_escalation | LinearSVC_C0.05
leave one out | q=0.25 | remove=hormuz_shipping | LinearSVC_C0.05
leave one out | q=0.25 | remove=sanctions_exports | LinearSVC_C0.05
leave 

,filter_q,analysis,removed_topic,model,n_features,accuracy,balanced_accuracy,auc,up_rate_true,up_rate_pred
12,0.25,all_core,none,LinearSVC_C0.05,65,0.594595,0.529279,0.604091,0.600000,0.832432
17,0.25,leave_one_out,sanctions_exports,LinearSVC_C0.05,52,0.637838,0.576577,0.610908,0.600000,0.821622
16,0.25,leave_one_out,hormuz_shipping,LinearSVC_C0.05,52,0.627027,0.569820,0.590090,0.600000,0.800000
15,0.25,leave_one_out,military_escalation,LinearSVC_C0.05,52,0.616216,0.551802,0.598856,0.600000,0.832432
14,0.25,leave_one_out,diplomacy_ceasefire,LinearSVC_C0.05,52,0.594595,0.520270,0.555028,0.600000,0.875676
13,0.25,leave_one_out,inventory_eia,LinearSVC_C0.05,52,0.589189,0.509009,0.543341,0.600000,0.902703
6,0.25,all_core,none,Logit_L1_C0.5,65,0.621622,0.556306,0.603238,0.600000,0.837838
11,0.25,leave_one_out,sanctions_exports,Logit_L1_C0.5,52,0.621622,0.563063,0.614804,0.600000,0.805405
9,0.25,leave_one_out,military_escalation,Logit_L1_C0.5,52,0.616216,0.551802,0.599221,0.600000,0.832432
10,0.25,leave_one_out,hormuz_shipping,Logit_L1_C0.5,52,0.594595,0.524775,0.593255,0.600000,0.854054


Saved: /content/drive/MyDrive/bigquery/experiment3b_leave_one_topic_out.csv


In [16]:
# =========================
# Cell 13: Leave-one-out delta
# 作用：
# 计算“删掉某个 topic 后，表现变化多少”。
#
# delta < 0：
# 删掉后变差，说明该 topic 有帮助。
#
# delta > 0：
# 删掉后变好，说明该 topic 可能是噪声。
# =========================

delta_rows = []

group_cols = ["filter_q", "model"]

for keys, group in leave_one_results.groupby(group_cols):
    filter_q, model_name = keys

    base = group[group["analysis"] == "all_core"]

    if len(base) == 0:
        continue

    base = base.iloc[0]
    loo = group[group["analysis"] == "leave_one_out"]

    for _, row in loo.iterrows():
        delta_rows.append({
            "filter_q": filter_q,
            "model": model_name,
            "removed_topic": row["removed_topic"],

            "base_accuracy": base["accuracy"],
            "loo_accuracy": row["accuracy"],
            "delta_accuracy": row["accuracy"] - base["accuracy"],

            "base_balanced_accuracy": base["balanced_accuracy"],
            "loo_balanced_accuracy": row["balanced_accuracy"],
            "delta_balanced_accuracy": row["balanced_accuracy"] - base["balanced_accuracy"],

            "base_auc": base["auc"],
            "loo_auc": row["auc"],
            "delta_auc": row["auc"] - base["auc"],

            "base_up_rate_pred": base["up_rate_pred"],
            "loo_up_rate_pred": row["up_rate_pred"],
        })

leave_one_delta = pd.DataFrame(delta_rows)

display(
    leave_one_delta[
        [
            "filter_q",
            "model",
            "removed_topic",
            "delta_accuracy",
            "delta_balanced_accuracy",
            "delta_auc",
            "base_accuracy",
            "loo_accuracy",
            "base_auc",
            "loo_auc"
        ]
    ].sort_values(["filter_q", "delta_auc"], ascending=[True, True])
)

leave_one_delta.to_csv(DATA_DIR / "experiment3b_leave_one_topic_out_delta.csv", index=False)
print("Saved:", DATA_DIR / "experiment3b_leave_one_topic_out_delta.csv")

,filter_q,model,removed_topic,delta_accuracy,delta_balanced_accuracy,delta_auc,base_accuracy,loo_accuracy,base_auc,loo_auc
10,0.25,Logit_L2_C0.1,inventory_eia,-0.005405,-0.009009,-0.066228,0.610811,0.605405,0.610421,0.544193
0,0.25,LinearSVC_C0.05,inventory_eia,-0.005405,-0.020270,-0.060750,0.594595,0.589189,0.604091,0.543341
5,0.25,Logit_L1_C0.5,inventory_eia,-0.048649,-0.049550,-0.060141,0.621622,0.572973,0.603238,0.543097
1,0.25,LinearSVC_C0.05,diplomacy_ceasefire,0.000000,-0.009009,-0.049063,0.594595,0.594595,0.604091,0.555028
11,0.25,Logit_L2_C0.1,diplomacy_ceasefire,-0.027027,-0.027027,-0.035184,0.610811,0.583784,0.610421,0.575237
6,0.25,Logit_L1_C0.5,diplomacy_ceasefire,-0.037838,-0.047297,-0.033723,0.621622,0.583784,0.603238,0.569515
3,0.25,LinearSVC_C0.05,hormuz_shipping,0.032432,0.040541,-0.014000,0.594595,0.627027,0.604091,0.590090
13,0.25,Logit_L2_C0.1,hormuz_shipping,-0.005405,0.004505,-0.011444,0.610811,0.605405,0.610421,0.598977
8,0.25,Logit_L1_C0.5,hormuz_shipping,-0.027027,-0.031532,-0.009983,0.621622,0.594595,0.603238,0.593255
2,0.25,LinearSVC_C0.05,military_escalation,0.021622,0.022523,-0.005235,0.594595,0.616216,0.604091,0.598856


Saved: /content/drive/MyDrive/bigquery/experiment3b_leave_one_topic_out_delta.csv


In [ ]:
# =========================
# Cell 14: Topic group sensitivity
# 作用：
# 测试不同 topic 组合。
#
# 例如：
# 1. geopolitical_risk
# 2. supply_disruption
# 3. inventory_only
# 4. diplomacy_only
# 5. conflict_vs_diplomacy
# =========================

topic_groups = {
    "all_core": [
        "inventory_eia",
        "diplomacy_ceasefire",
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports"
    ],

    "geopolitical_risk": [
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports",
        "diplomacy_ceasefire"
    ],

    "supply_disruption": [
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports"
    ],

    "deescalation_only": [
        "diplomacy_ceasefire"
    ],

    "inventory_only": [
        "inventory_eia"
    ],

    "conflict_vs_diplomacy": [
        "military_escalation",
        "diplomacy_ceasefire"
    ],

    "inventory_plus_geopolitics": [
        "inventory_eia",
        "military_escalation",
        "hormuz_shipping",
        "sanctions_exports"
    ],
}

group_rows = []

group_filter_q_list = [0.25, 0.30, 0.40]
group_models = [
    "Logit_L2_C0.1",
    "Logit_L1_C0.5",
    "LinearSVC_C0.05"
]

for filter_q in group_filter_q_list:
    df_run, return_threshold = make_meaningful_df(df_feat, filter_q=filter_q)

    for group_name, topics in topic_groups.items():
        cols = combine_topic_cols(topics, use_shock=True)

        if len(cols) == 0:
            continue

        for model_name in group_models:
            print(f"topic group | q={filter_q} | {group_name} | {model_name}")

            try:
                res = evaluate_walkforward(
                    df_run,
                    feature_cols=cols,
                    model_name=model_name,
                    n_splits=5
                )

                res["filter_q"] = filter_q
                res["return_threshold"] = return_threshold
                res["topic_group"] = group_name
                res["topics"] = ",".join(topics)
                group_rows.append(res)

            except Exception as e:
                print("Failed:", filter_q, group_name, model_name, e)

topic_group_results = pd.DataFrame(group_rows)

topic_group_results = topic_group_results.sort_values(
    ["filter_q", "accuracy", "balanced_accuracy", "auc"],
    ascending=[True, False, False, False]
)

display(
    topic_group_results[
        [
            "filter_q",
            "topic_group",
            "topics",
            "model",
            "n_features",
            "accuracy",
            "balanced_accuracy",
            "auc",
            "up_rate_true",
            "up_rate_pred"
        ]
    ].head(80)
)

topic_group_results.to_csv(DATA_DIR / "experiment3c_topic_group_sensitivity.csv", index=False)
print("Saved:", DATA_DIR / "experiment3c_topic_group_sensitivity.csv")